# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [1]:
# Carga de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

housing = pd.read_csv("../data/interim/train_set.csv")

### 1. Valores nulos

In [ ]:
# Corregimos los valores nulos de la columna `total_bedrooms`
housing["total_bedrooms"].describe()




count    16344.000000
mean       538.949094
std        423.862079
min          1.000000
25%        296.000000
50%        434.000000
75%        645.000000
max       6210.000000
Name: total_bedrooms, dtype: float64

In [3]:
# Calculamos la mediana
median_bedrooms = housing["total_bedrooms"].median()
print("Mediana:", median_bedrooms)

Mediana: 434.0


In [4]:
# Rellenamos los valores nulos con la mediana, por que tiene valores maximos muy alejados del promedio
housing["total_bedrooms"] = housing["total_bedrooms"].fillna(median_bedrooms)

# Validamos que no existan más valores nulos
print(housing.isnull().sum())

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


### 2. Codificación Categórica

In [5]:
# Mostramos los valores de la columna `ocean_proximity`
print(housing["ocean_proximity"].value_counts())

ocean_proximity
<1H OCEAN     7274
INLAND        5301
NEAR OCEAN    2089
NEAR BAY      1846
ISLAND           2
Name: count, dtype: int64


In [6]:
# Realizamos la codificacion nominal de la columna
housing = pd.get_dummies(housing, columns=["ocean_proximity"], drop_first=True)

In [7]:
# Validamos la nueva estructura del DataFrame
display(housing.head())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,False,False,True,False
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,False,False,False,False
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,True,False,False,False
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,True,False,False,False
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,False,False,False,True


### 3. Feature Engineering

In [8]:
# Creamos nuevas características a partir de las existentes

# Agregamos las columnas sugeridas
housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"] / housing["households"]


In [9]:
# Agregamos la distancia a San Francisco y a Los Angeles

# Coordenadas aproximadas
SF_LAT, SF_LON = 37.77, -122.42
LA_LAT, LA_LON = 34.05, -118.24

housing["dist_sf"] = np.sqrt(
    (housing["latitude"] - SF_LAT)**2 + (housing["longitude"] - SF_LON)**2
)
housing["dist_la"] = np.sqrt(
    (housing["latitude"] - LA_LAT)**2 + (housing["longitude"] - LA_LON)**2
)


In [10]:
# Agregamos el ingreso por habitación
housing["income_per_room"] = housing["median_income"] / housing["rooms_per_household"]

In [15]:
# Verificamos las nuevas características
display(housing[["rooms_per_household", "bedrooms_per_room", "population_per_household", 
                 "dist_sf", "dist_la", "income_per_room"]].head())  

,rooms_per_household,bedrooms_per_room,population_per_household,dist_sf,dist_la,income_per_room
0,3.211799,0.335742,1.524178,0.030000,5.615594,0.653434
1,5.504202,0.180153,1.865546,5.431252,0.166433,1.105991
2,5.334975,0.200369,2.768473,0.736003,5.706461,0.456047
3,5.351282,0.203881,2.365385,6.660068,1.169145,0.422665
4,3.725256,0.277371,1.631399,5.850889,0.294109,0.947371


In [17]:
# Verificamos las nuevas características
display(housing[["rooms_per_household", "bedrooms_per_room", "population_per_household", 
                 "dist_sf", "dist_la", "income_per_room"]].describe())  

,rooms_per_household,bedrooms_per_room,population_per_household,dist_sf,dist_la,income_per_room
count,16512.000000,16512.000000,16512.000000,16512.000000,16512.000000,16512.000000
mean,5.441010,0.213727,2.995974,3.867050,2.653804,0.714828
std,2.574143,0.066077,4.457373,2.500086,2.410136,0.254789
min,0.888889,0.037066,0.692308,0.000000,0.000000,0.013213
25%,4.443636,0.175058,2.433426,1.205062,0.321403,0.542117
50%,5.235573,0.203120,2.822316,5.255730,1.709415,0.706884
75%,6.053843,0.239844,3.286385,5.842527,5.206256,0.860954
max,141.909091,2.818182,502.461538,9.307943,9.869534,5.168025


### 4. Escalado de Variables

In [12]:
# Separamos la funcion objetivo de las características
X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]

# Escalamos las características numéricas
numerical_cols = X.select_dtypes(include=["float64", "int64"]).columns

scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

In [13]:
# Validamos el escalado
display(X[numerical_cols].head())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,rooms_per_household,bedrooms_per_room,population_per_household,dist_sf,dist_la,income_per_room
0,-1.423037,1.013606,1.861119,0.311912,1.368167,0.137460,1.394812,-0.936491,-0.866027,1.846624,-0.330204,-1.534814,1.228926,-0.240964
1,0.596394,-0.702103,0.907630,-0.308620,-0.435925,-0.693771,-0.373485,1.171942,0.024550,-0.508121,-0.253616,0.625679,-1.032077,1.535294
2,-1.203098,1.276119,0.351428,-0.712240,-0.760709,-0.788768,-0.775727,-0.759789,-0.041193,-0.202155,-0.051041,-1.252414,1.266629,-1.015698
3,1.231216,-0.884924,-0.919891,0.702262,0.742306,0.383175,0.731375,-0.850281,-0.034858,-0.149006,-0.141475,1.117203,-0.616025,-1.146720
4,0.711362,-0.875549,0.589800,0.790125,1.595753,0.444376,1.755263,-0.180365,-0.666554,0.963208,-0.306148,0.793532,-0.979101,0.912719


## 5. Conclusiones de Limpieza y Enriquecimiento

#### Valores nulos
- La única columna con valores nulos era `total_bedrooms` con **168 registros vacíos**
- Se rellenaron con la **mediana (434)** en lugar del promedio porque la columna tiene valores extremos de hasta 6,210 que jalan el promedio hacia arriba
- Después del relleno el dataset quedó **completamente limpio**, sin ningún nulo

#### Codificación categórica
- La columna `ocean_proximity` tenía **5 categorías**: `<1H OCEAN` (la más común con 7,274 distritos), `INLAND`, `NEAR OCEAN`, `NEAR BAY` e `ISLAND` (solo 2 casos)
- Se usó **codificación nominal (One-Hot)** porque no existe un orden lógico entre las categorías — estar cerca del océano no es "mayor" que estar en  a 1 hora del océano, por ejemplo
- Se usó `drop_first=True` para evitar redundancia: si todas las dummies son False, el distrito pertenece a `<1H OCEAN`

#### Feature Engineering — Variables sugeridas
- **`rooms_per_household`**: promedio de 5 cuartos por hogar, más útil que el total  bruto de cuartos del distrito
- **`bedrooms_per_room`**: en promedio el 20-33% de los cuartos son dormitorios, lo que ayuda a identificar casas pequeñas vs. espaciosas
- **`population_per_household`**: promedio de 1.5-2.7 personas por hogar, indicador de densidad familiar por distrito

#### Feature Engineering — Variables propuestas
- **`dist_sf` y `dist_la`**: el distrito más cercano a SF tiene distancia 0.03 y el más lejano 6.6  lo que confirma quela proximidad a estas ciudades es un factor clave en el precio
- **`income_per_room`**: varía entre 0.42 y 1.10 en las primeras filas, combinando las dos señales más fuertes del EDA (ingreso y habitaciones por hogar)

#### Estado final del dataset
- Se pasó de **10 columnas originales** a **19 columnas** listas para modelado Todas las variables son numéricas, sin nulos, y representan mejor la realidad del distrito que los datos crudos originales
